## Day 1 task: implementing encode/decode 

In [1]:
B32 = "0123456789bcdefghjkmnpqrstuvwxyz"

def encode(lat: float, lon: float, precision: int = 7) -> str: 
    'Encode latitude and longitude into a geohash string with the specified precision.'
    lat_r, lon_r, bits, out, even = [-90.0, 90.0], [-180.0, 180.0], 0, "", True
    for i in range(precision * 5):
        rng, val = (lon_r, lon) if even else (lat_r, lat)
        mid = (rng[0] + rng[1]) / 2
        bits = bits * 2 + (val >= mid)
        rng[0 if val >= mid else 1] = mid
        even = not even
        if i % 5 == 4:
            out, bits = out + B32[bits], 0
    return out


def decode(gh: str) -> tuple[float, float, float, float]: 
    'Decode a geohash string into latitude and longitude bounds.'
    lat_r, lon_r, even = [-90.0, 90.0], [-180.0, 180.0], True
    for ch in gh:
        v = B32.index(ch)
        for k in range(4, -1, -1):
            rng = lon_r if even else lat_r
            mid = (rng[0] + rng[1]) / 2
            rng[0 if (v >> k) & 1 else 1] = mid
            even = not even
    return lat_r, lon_r

print('encode: ', encode(42.6, -5.6, 5), '\n'
      'decode: ', decode("ezs42"))

encode:  ezs42 
decode:  ([42.5830078125, 42.626953125], [-5.625, -5.5810546875])


## Round-trip check
Random points at random precisions: each point must lie inside its decoded cell, and the cell centre must encode back to the same geohash.

In [2]:
import random

random.seed(42)
N = 10_000
for _ in range(N):
    lat, lon = random.uniform(-90, 90), random.uniform(-180, 180)
    p = random.randint(1, 12)
    gh = encode(lat, lon, p)
    (lat_min, lat_max), (lon_min, lon_max) = decode(gh)
    assert len(gh) == p, gh
    assert lat_min <= lat <= lat_max and lon_min <= lon <= lon_max, (lat, lon, gh)
    assert encode((lat_min + lat_max) / 2, (lon_min + lon_max) / 2, p) == gh
print(f"round trip OK: {N:,} random points, precision 1-12")

round trip OK: 10,000 random points, precision 1-12


## The edge problem
Geohash orders cells along a Z-order curve. Two points a few metres apart can sit in different cells, and where the curve jumps they share a much shorter prefix, sometimes none at all. A prefix search alone misses them.

In [3]:
import math
from os.path import commonprefix

import numpy as np


def haversine_m(lat1, lon1, lat2, lon2, r=6_371_008.8):
    """Great-circle distance in metres; works on scalars and numpy arrays."""
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp, dl = p2 - p1, np.radians(np.asarray(lon2) - lon1)
    h = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(h))


def show_pair(name, a, b, p=7):
    ga, gb = encode(*a, p), encode(*b, p)
    shared = len(commonprefix([ga, gb]))
    print(f"{name}: {haversine_m(*a, *b):5.1f} m apart  {ga} vs {gb}  shared prefix: {shared} of {p}")


show_pair("London, across Greenwich  ", (51.5, -0.0001), (51.5, 0.0001))

# Almaty: two points 10 m apart on either side of the east edge of a precision-7 cell
LAT, LON = 43.2389, 76.9286
_, (_, east) = decode(encode(LAT, LON, 7))
dlon = 5 / (111_320 * math.cos(math.radians(LAT)))  # 5 m in degrees of longitude at this latitude
A, B = (LAT, east - dlon), (LAT, east + dlon)
show_pair("Almaty, one cell edge     ", A, B)

# Worst edge in the city: every precision-7 cell edge along this latitude, across the Almaty bbox
step = 360 / 2**18  # precision 7 = 35 bits, 18 of them longitude
edges = [k * step - 180 for k in range(math.ceil((76.74 + 180) / step), math.floor((77.16 + 180) / step) + 1)]
worst = min(edges, key=lambda e: len(commonprefix([encode(LAT, e - dlon, 7), encode(LAT, e + dlon, 7)])))
show_pair("Almaty, worst edge        ", (LAT, worst - dlon), (LAT, worst + dlon))

London, across Greenwich  :  13.8 m apart  gcpuzzr vs u10hbp2  shared prefix: 0 of 7
Almaty, one cell edge     :  10.0 m apart  txwtyx3 vs txwtyx6  shared prefix: 6 of 7
Almaty, worst edge        :  10.0 m apart  txwtzzr vs txwvbp2  shared prefix: 3 of 7


## The fix: query the 8 neighbours too
Take the cell centre, step one cell height/width in each direction and re-encode at the same precision. Longitude wraps at ±180; a cell on a pole has nothing beyond it, so it gets 5 neighbours.

In [4]:
def neighbours(gh: str) -> list[str]:
    """The cells around gh at the same precision, in order N, NE, E, SE, S, SW, W, NW."""
    (lat_min, lat_max), (lon_min, lon_max) = decode(gh)
    h, w = lat_max - lat_min, lon_max - lon_min
    lat_c, lon_c = lat_min + h / 2, lon_min + w / 2
    out = []
    for dy, dx in [(1, 0), (1, 1), (0, 1), (-1, 1), (-1, 0), (-1, -1), (0, -1), (1, -1)]:
        lat = lat_c + dy * h
        if not -90 < lat < 90:  # nothing beyond the poles
            continue
        lon = (lon_c + dx * w + 180) % 360 - 180  # wrap across the antimeridian
        out.append(encode(lat, lon, len(gh)))
    return out


cell_a = encode(*A, 7)
print(cell_a, "->", neighbours(cell_a))
print("B in A's cell:", encode(*B, 7) == cell_a, "| B in A's 9 cells:", encode(*B, 7) in neighbours(cell_a))

# Checks on random cells: right count, all distinct, each one touches the centre cell
random.seed(1)
for _ in range(5000):
    gh = encode(random.uniform(-90, 90), random.uniform(-180, 180), random.randint(1, 9))
    (a0, a1), (o0, o1) = decode(gh)
    nbs = neighbours(gh)
    assert len(set(nbs)) == (5 if a0 == -90 or a1 == 90 else 8) and gh not in nbs, gh
    for n in nbs:
        (b0, b1), (p0, p1) = decode(n)
        assert max(a0, b0) <= min(a1, b1), (gh, n)  # lat ranges meet
        assert max(o0, p0) <= min(o1, p1) or {o0, p1} == {-180, 180} or {o1, p0} == {-180, 180}, (gh, n)
print("neighbours OK: 5,000 random cells")
print("antimeridian:", encode(0, 179.9999, 5), "E neighbour ->", neighbours(encode(0, 179.9999, 5))[2])

txwtyx3 -> ['txwtyx9', 'txwtyxd', 'txwtyx6', 'txwtyx4', 'txwtyx1', 'txwtyx0', 'txwtyx2', 'txwtyx8']
B in A's cell: False | B in A's 9 cells: True


neighbours OK: 5,000 random cells
antimeridian: xbpbp E neighbour -> 80000


## Does it matter on real data? Almaty places, 100 m radius
Every Overture place from Module 0 (`places_flat.parquet`) gets a precision-7 geohash. For 500 random centres: the true set within 100 m (exact haversine) vs the candidates from the centre's cell alone and from the 9 cells. A 100 m circle fits inside the 9 cells only because a precision-7 cell here is larger than 100 m on both sides.

In [5]:
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
places = pd.read_parquet(ROOT / "data" / "processed" / "places_flat.parquet")  # Module 0, day 5

P, R = 7, 100
(c0, c1), (d0, d1) = decode(encode(LAT, LON, P))
cell_h, cell_w = haversine_m(c0, LON, c1, LON), haversine_m(LAT, d0, LAT, d1)
print(f"precision-{P} cell in Almaty: {cell_h:.0f} m tall x {cell_w:.0f} m wide; radius {R} m")
assert R <= min(cell_h, cell_w), "radius larger than a cell: 9 cells no longer cover the circle"

places["gh"] = [encode(la, lo, P) for la, lo in zip(places.lat, places.lon)]
by_cell = places.groupby("gh").indices  # geohash -> row positions
lat, lon = places.lat.to_numpy(), places.lon.to_numpy()

rows = []
for c in places.sample(500, random_state=42).itertuples():
    truth = set(np.flatnonzero(haversine_m(c.lat, c.lon, lat, lon) <= R))
    one = set(by_cell.get(c.gh, []))
    nine = set().union(*(by_cell.get(g, []) for g in [c.gh, *neighbours(c.gh)]))
    rows.append({"truth": len(truth), "candidates_1": len(one), "candidates_9": len(nine),
                 "recall_1": len(truth & one) / len(truth), "recall_9": len(truth & nine) / len(truth)})
res = pd.DataFrame(rows)

print(f"recall, centre cell only: mean {res.recall_1.mean():.1%}, centres missing something {(res.recall_1 < 1).mean():.0%}")
print(f"recall, 9 cells:          mean {res.recall_9.mean():.1%}, worst {res.recall_9.min():.0%}")
print(f"places per query: truth median {res.truth.median():.0f}, 9-cell candidates median {res.candidates_9.median():.0f}"
      f" ({(res.truth / res.candidates_9).median():.0%} of candidates are true hits, so re-rank by exact distance)")
assert (res.recall_9 == 1).all()

precision-7 cell in Almaty: 153 m tall x 111 m wide; radius 100 m


recall, centre cell only: mean 63.4%, centres missing something 84%
recall, 9 cells:          mean 100.0%, worst 100%
places per query: truth median 14, 9-cell candidates median 51 (33% of candidates are true hits, so re-rank by exact distance)


Map: A and B 10 m apart, A's cell (orange) and its 8 neighbours (blue).

In [6]:
import folium


def cell_box(gh):
    (la0, la1), (lo0, lo1) = decode(gh)
    return [[la0, lo0], [la1, lo1]]


m = folium.Map(location=A, zoom_start=17)
for g in neighbours(cell_a):
    folium.Rectangle(cell_box(g), color="#1F5E96", weight=1, fill=True, fill_opacity=0.08, tooltip=g).add_to(m)
folium.Rectangle(cell_box(cell_a), color="#C4561D", weight=2, fill=True, fill_opacity=0.2, tooltip=cell_a).add_to(m)
for name, pt in [("A", A), ("B", B)]:
    folium.CircleMarker(pt, radius=5, color="black", fill=True, tooltip=f"{name}: {encode(*pt, 7)}").add_to(m)
m